# Trabalho de IA - Resolução do Jogo do Labirinto usando Q- Learning Aproximado 

### Baixando as bibliotecas necessárias

In [17]:
%pip install matplotlib numpy
%pip install pandas 

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Importando as bibliotecas

In [18]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd 

In [19]:
# Função que calcula a posição candidata. 
# Ela recebe a posição atual e a ação e, 
# a partir disso, calcula a posição candidata. 
def calcular_posicao_candidata(posicao_atual, acao):

    linha = posicao_atual[0]
    coluna = posicao_atual[1]

    if acao == "cima":

        return (linha - 1, coluna)

    if acao == "baixo":

        return (linha + 1, coluna)

    if acao == "direita":

        return (linha, coluna + 1)

    if acao == "esquerda":

        return (linha, coluna - 1)

    return posicao_atual # Se o agente recebe uma ação inválida, ele fica parado.

In [20]:
# Função que recebe a posição candidata e o mapa do labirinto e 
# verifica se a posição está dentro dos limites do mapa. 
def posicao_eh_valida(posicao_candidata, mapa_labirinto):

    qtd_linhas_mapa = len(mapa_labirinto)
    qtd_colunas_mapa = len(mapa_labirinto[0])

    linha = posicao_candidata[0]
    coluna = posicao_candidata[1]

    if ((0 <= linha < qtd_linhas_mapa) and (0 <= coluna < qtd_colunas_mapa)):

        return True

    return False 

# Função que recebe a posição candidata e o mapa do labirinto e 
# verifica se a posição é uma parede. 
def posicao_eh_parede(posicao_candidata, mapa_labirinto):

    linha = posicao_candidata[0]
    coluna = posicao_candidata[1]

    if mapa_labirinto[linha][coluna] == "#":

        return True 

    return False

# Função que recebe a posição candidata e o mapa do labirinto e 
# verifica se a posição é uma armadilha. 
def posicao_eh_armadilha(posicao_candidata, mapa_labirinto):

    linha = posicao_candidata[0]
    coluna = posicao_candidata[1]

    if mapa_labirinto[linha][coluna] == "T":

        return True 

    return False

# Função que recebe a posição candidata e o mapa do labirinto e 
# verifica se a posição é uma parede. 
def posicao_eh_saida(posicao_candidata, mapa_labirinto):

    linha = posicao_candidata[0]
    coluna = posicao_candidata[1]

    if mapa_labirinto[linha][coluna] == "E":

        return True 

    return False


In [21]:
# Função que recebe a posição atual, a ação e o mapa do labirinto e 
# devolve uma tupla que informa qual será a nova posição do agente, 
# qual será a recompensa dele para a ação e informa se ele chegou à saída.
def movimentar(posicao_atual, acao, mapa_labirinto):

    posicao_candidata = calcular_posicao_candidata(posicao_atual, acao)

    # A posição candidata está além dos limites do mapa, logo 
    # o agente não se mexe, tem penalidade de -3 e não chegou à saída. 
    if not posicao_eh_valida(posicao_candidata, mapa_labirinto):

        return (posicao_atual, -3, False)

    # A posição candidata é uma parede , logo o agente não se 
    # mexe, tem penalidade de -3 e não chegou à saída.
    if posicao_eh_parede(posicao_candidata, mapa_labirinto):

        return (posicao_atual, -3, False)

    # A posição candidata é uma armadilha, logo o agente 
    # avança, tem penalidade de -10 e não chegou à saída.
    if posicao_eh_armadilha(posicao_candidata, mapa_labirinto):

        return (posicao_candidata, -10, False)

    # A posição candidata é a saída, logo o agente 
    # avança para ela, tem recompensa de +30 e chegou à saída.
    if posicao_eh_saida(posicao_candidata, mapa_labirinto):

        return (posicao_candidata, +30, True)

    # É um simples movimento em uma célula livre. 
    else:

        return (posicao_candidata, -1, False)

In [22]:
%matplotlib inline
from IPython.display import display

In [23]:
# Função que ao receber o mapa do labirinto e a posição atual, desenha 
# o jogo. 
def desenhar_jogo(posicao_atual, mapa_labirinto):
    
    # Criamos um dicionário para mapear os símbolos do mapa 
    # à números a fim de facilitar o preenchimento com cor. 
    mapeamento_numerico = {

        ".": 0,  # Caminho livre
        "#": 1,  # Parede
        "S": 2,  # Início
        "E": 3,  # Saída
        "T": 4   # Armadilha
    }
    
    # Convertemos a matriz de strings para uma matriz numérica
    linhas = len(mapa_labirinto)
    colunas = len(mapa_labirinto[0])
    matriz_numerica = np.zeros((linhas, colunas))
    
    # Preenchemos a matriz numérica com os valores do dicionário. 
    for l in range(linhas):

        for c in range(colunas):

            simbolo = mapa_labirinto[l][c]

            matriz_numerica[l][c] = mapeamento_numerico.get(simbolo, 0)
            
    figura = plt.figure(figsize=(8, 6))
    
    # Criamos um mapeamento entre as cores e os símbolos do mapa:
    # 0=Branco (Caminho), 1=Preto (Parede), 2=Azul (Início), 3=Verde (Saída), 4=Vermelho (Armadilha)
    cmap_customizado = plt.cm.colors.ListedColormap(['white', 'black', 'blue', 'green', 'red'])
    
    plt.imshow(matriz_numerica, cmap=cmap_customizado, origin='upper', vmin=0, vmax=4)
    
    # Desenhando a posição atual do agente
    linha_jogador, coluna_jogador = posicao_atual
    
    marcador, = plt.plot(coluna_jogador, linha_jogador, marker='o', color='gold', markersize=15, markeredgecolor='black', label='Jogador')
    
    plt.xticks(range(colunas))
    plt.yticks(range(linhas))
    plt.grid(True, which='both', color='gray', linestyle='-', linewidth=0.5)
    
    # plt.show(block=False)

    return figura, marcador

In [24]:
# Função que recebe a posição atual, ação, mapa e posição de saída 
# e extrai as caracterpisticas. 
def extrair_caracteristicas(posicao_atual, acao, mapa_labirinto, posicao_saida):

    posicao_candidata = calcular_posicao_candidata(posicao_atual, acao)

    colisao = 0.0
    armadilha = 0.0
    saida = 0.0 

    if not posicao_eh_valida(posicao_candidata, mapa_labirinto) or posicao_eh_parede(posicao_candidata, mapa_labirinto):

        posicao_resultante = posicao_atual

        colisao = 1.0 

    else:
        
        posicao_resultante = posicao_candidata

        if posicao_eh_armadilha(posicao_candidata, mapa_labirinto):

            armadilha = 1.0 

        if posicao_eh_saida(posicao_candidata, mapa_labirinto):

            saida = 1.0 

    # Vamos calcular a distância normalizada até a saída usando a distância
    # de Manhattan, pois se trata de uma grade. 

    distancia_absoluta = abs(posicao_resultante[0] - posicao_saida[0]) + abs(posicao_resultante[1] - posicao_saida[1])

    max_distancia_possivel = (len(mapa_labirinto) - 1) + (len(mapa_labirinto[0]) - 1)

    distancia_normalizada = distancia_absoluta / max_distancia_possivel

    bias = 1.0 # Ajuda nos cálculos da IA 

    caracteristicas_lista = [bias, colisao, armadilha, saida, distancia_normalizada]

    # Só a distância até a saída não mostra em qual corredor o agente está.
    # Por isso, usamos bases gaussianas para representar a proximidade a
    # alguns pontos do mapa. Assim, ele pode aprender a avaliar regiões
    # diferentes, mesmo quando elas estão à mesma distância da saída.

    # Definindo 12 pontos de referência para calcularmos as bases gaussianas. 
    linhas_referencia = [1, 3, 5]
    colunas_referencia = [1, 3, 5, 7]

    linha_res = posicao_resultante[0]
    coluna_res = posicao_resultante[1]

    # Calculando as bases gaussianas. 
    for linha in linhas_referencia:

        for coluna in colunas_referencia:

            distancia_quadrada = (linha_res - linha) ** 2 + (coluna_res - coluna) ** 2
            
            proximidade = np.exp(-distancia_quadrada / 4)

            caracteristicas_lista.append(proximidade)

    vetor_caracteristicas = np.array(caracteristicas_lista, dtype=np.float64)

    return vetor_caracteristicas

In [25]:
# Função que calula o q_valor aproximado realizando o produto 
# escalar entre os pesos e as caracteristicas. 
def calcular_q(pesos, vetor_caracteristicas):

    return np.dot(pesos, vetor_caracteristicas)

In [26]:
# Define qual ação o agente vai tomar usando a estratégia epsilon-greedy.
# Tem uma chance (epsilon) de testar direções aleatórias para explorar o labirinto,
# senão ele escolhe a ação com maior valor estimado de Q.
def escolher_acao(posicao_atual, posicao_saida, mapa_labirinto, epsilon, acoes_disponiveis, gerador_aleatorio, pesos):

    # Sortei um número entre 0 e 1 
    if gerador_aleatorio.random() < epsilon:

        # O agente vai explorar. Ele escolhe uma das ações com igual probabilidade. 
        return gerador_aleatorio.choice(acoes_disponiveis)

    # O agente resolve aproveitar e vai no maior Q para pontuar mais.
    # Ele calcula o q de cada ação disponível e guarda em um diconário. 
    # Pega a ação ou as ações que atingiram o maior q e sorteia uma aleatoriamente.  
    valores_q = {}

    for acao in acoes_disponiveis:

        caracteristicas = extrair_caracteristicas(posicao_atual, acao, mapa_labirinto, posicao_saida)

        q_atual = calcular_q(pesos, caracteristicas)

        valores_q[acao] = q_atual

    maior_q = max(valores_q.values())

    acoes_maior_q = []

    for acao, q in valores_q.items():

        if q == maior_q:

            acoes_maior_q.append(acao)

    return gerador_aleatorio.choice(acoes_maior_q)

            


In [27]:
# Atualiza os pesos usando Q-Learning
def atualizar_pesos(pesos_atuais, caracteristicas_atuais, recompensa, 
                    proxima_posicao, chegou_saida, mapa_labirinto, alpha, gama, acoes_disponiveis, posicao_saida):

    # Calculando a previsão anterior. 
    q_atual = calcular_q(pesos_atuais, caracteristicas_atuais)

    # Calculando o melhor valor futuro 
    if chegou_saida:

        melhor_q_futuro = 0.0 # Se chegou à saída, não tem próxima ação. 

    else:

        q_futuros = []

        for acao in acoes_disponiveis:

            caracteristicas_futuras = extrair_caracteristicas(proxima_posicao, acao, mapa_labirinto, posicao_saida)

            q_futuro = calcular_q(pesos_atuais, caracteristicas_futuras)

            q_futuros.append(q_futuro)

        melhor_q_futuro = max(q_futuros)

    novo_alvo = recompensa + gama * melhor_q_futuro

    erro_temporal = novo_alvo - q_atual

    novos_pesos = pesos_atuais + alpha * erro_temporal * caracteristicas_atuais

    return novos_pesos



In [28]:
def treinar_episodio(mapa_labirinto, pesos, alpha, gama, epsilon, acoes_disponiveis, gerador_aleatorio, limite_movimentos, posicao_inicial, posicao_saida):

    posicao_atual = posicao_inicial
    pontuacao_acumulada = 0
    contador_movimentos = 0

    chegou_saida = posicao_eh_saida(posicao_atual, mapa_labirinto)

    while ((chegou_saida != True) and (contador_movimentos < limite_movimentos)):

        acao_escolhida = escolher_acao(posicao_atual, posicao_saida, mapa_labirinto, epsilon, acoes_disponiveis, gerador_aleatorio, pesos)

        caracteristicas = extrair_caracteristicas(posicao_atual, acao_escolhida, mapa_labirinto, posicao_saida)

        proxima_posicao, recompensa, chegou_saida = movimentar(posicao_atual, acao_escolhida, mapa_labirinto)

        pesos = atualizar_pesos(pesos, caracteristicas, recompensa, proxima_posicao, chegou_saida, mapa_labirinto, alpha, gama, acoes_disponiveis, posicao_saida)

        posicao_atual = proxima_posicao

        pontuacao_acumulada += recompensa

        contador_movimentos += 1

    return pesos, pontuacao_acumulada, contador_movimentos, chegou_saida


In [29]:
def treinar_agente(mapa_labirinto, alpha, gama, epsilon, acoes_disponiveis, limite_movimentos, posicao_inicial, posicao_saida, seed, qtd_episodios):

    pesos = np.zeros(17, dtype=np.float64)

    gerador_aleatorio = np.random.default_rng(seed)

    episodio = 0 

    resultados = []

    while episodio < qtd_episodios:

        pesos, pontuacao_acumulada, contador_movimentos, chegou_saida = treinar_episodio(mapa_labirinto, pesos, alpha, gama, epsilon, acoes_disponiveis, gerador_aleatorio, limite_movimentos, posicao_inicial, posicao_saida)

        resultados.append({

            "episodio": episodio + 1, 
            "pontuacao": pontuacao_acumulada,
            "movimentos": contador_movimentos,
            "chegou_saida": chegou_saida
        })

        episodio += 1

    return pesos, resultados

    

In [30]:
def avaliar_agente(pesos_finais, mapa_labirinto, posicao_inicial, posicao_saida, acoes_disponiveis, gerador_aleatorio, limite_movimentos):

    posicao_atual = posicao_inicial
    pontuacao_acumulada = 0
    contador_movimentos = 0

    chegou_saida = posicao_eh_saida(posicao_atual, mapa_labirinto)

    posicoes_percorridas = [posicao_inicial]

    while not chegou_saida and contador_movimentos < limite_movimentos:

        acao = escolher_acao(posicao_atual, posicao_saida, mapa_labirinto, 0, acoes_disponiveis, gerador_aleatorio, pesos_finais)

        posicao_atual, recompensa, chegou_saida = movimentar(posicao_atual, acao, mapa_labirinto)

        pontuacao_acumulada += recompensa

        contador_movimentos += 1

        posicoes_percorridas.append(posicao_atual)

    return pontuacao_acumulada, contador_movimentos, chegou_saida, posicoes_percorridas



In [31]:
def comparar_alphas(alphas, seeds, mapa_labirinto, gama, epsilon, acoes_disponiveis, limite_movimentos, posicao_inicial, posicao_saida, qtd_episodios):

    resultados = []

    for alpha in alphas:

        for seed in seeds:

            pesos_finais, historico_treinamento = treinar_agente(mapa_labirinto, alpha, gama, epsilon, acoes_disponiveis, limite_movimentos, posicao_inicial, posicao_saida, seed, qtd_episodios)

            # criamos um gerador de avaliação geral que vai ser usado nos 3 modelos. 
            gerador_avaliacao = np.random.default_rng(123)

            pontuacao_acumulada, contador_movimentos, chegou_saida, posicoes_percorridas = avaliar_agente(pesos_finais, mapa_labirinto, posicao_inicial, posicao_saida, acoes_disponiveis, gerador_avaliacao, limite_movimentos)

            resultados.append({

                "alpha": alpha,
                "seed": seed,
                "episodios": qtd_episodios,
                "pontuacao_acumulada": pontuacao_acumulada,
                "qtd_movimentos": contador_movimentos,
                "chegou_saida": chegou_saida,

            })

    return resultados

            

In [ ]:
def comparar_episodios(lista_episodios, seeds, mapa_labirinto, alpha, gama, epsilon, acoes_disponiveis, limite_movimentos, posicao_inicial, posicao_saida):

    resultados = []
    
    for qtd_episodios in lista_episodios:
    
        for seed in seeds:
    
            pesos_finais, historico_treinamento = treinar_agente(mapa_labirinto, alpha, gama, epsilon, acoes_disponiveis, limite_movimentos, posicao_inicial, posicao_saida, seed, qtd_episodios)
    
            # criamos um gerador de avaliação geral que vai ser usado nos 3 modelos. 
            gerador_avaliacao = np.random.default_rng(123)
    
            pontuacao_acumulada, contador_movimentos, chegou_saida, posicoes_percorridas = avaliar_agente(pesos_finais, mapa_labirinto, posicao_inicial, posicao_saida, acoes_disponiveis, gerador_avaliacao, limite_movimentos)
    
            resultados.append({
    
                "episodios": qtd_episodios,
                "seed": seed,
                "alpha": alpha,
                "pontuacao_acumulada": pontuacao_acumulada,
                "qtd_movimentos": contador_movimentos,
                "chegou_saida": chegou_saida,
    
            })
    
    return resultados

In [32]:
if __name__ == "__main__":

    mapa_labirinto = [
        ["#", "#", "#", "#", "#", "#", "#", "#", "#"],
        ["#", "S", ".", ".", ".", "#", ".", "E", "#"],
        ["#", ".", "#", "#", "T", "#", ".", "#", "#"],
        ["#", ".", "#", ".", ".", ".", ".", "#", "#"],
        ["#", ".", "#", ".", "#", "#", ".", "#", "#"],
        ["#", ".", ".", ".", "#", "T", ".", ".", "#"],
        ["#", "#", "#", "#", "#", "#", "#", "#", "#"]
    ]

    posicao_inicial = (1, 1)
    posicao_saida = (1, 7)
    posicao_armadilhas = [(2, 4), (5, 5)]

    valores_alpha = [0.005, 0.01, 0.05]
    seeds = [42, 43, 44]
    qtd_episodios = 1000
    gama = 0.95
    epsilon = 0.3
    limite_movimentos = 150

    acoes_disponiveis = ["cima", "baixo", "direita", "esquerda"]


    print("--- INÍCIO DA COMPARAÇÃO DOS ALPHAS ---")

    resultados_comparacao = comparar_alphas(valores_alpha, seeds, mapa_labirinto, gama, epsilon, acoes_disponiveis, limite_movimentos, posicao_inicial, posicao_saida, qtd_episodios)

    df_comparacao = pd.DataFrame(resultados_comparacao)

    print("\n--- RESULTADOS DA COMPARAÇÃO ---")
    print(df_comparacao.to_string(index=False))
    print(f"\nQuantidade de registros: {len(df_comparacao)}")

    df_resumo_alphas = df_comparacao.groupby("alpha").agg(
        taxa_sucesso=('chegou_saida', lambda x: x.astype(int).mean() * 100),
        pontuacao_media=('pontuacao_acumulada', 'mean'),
        pontuacao_desvio=('pontuacao_acumulada', 'std'),
        movimentos_medios=('qtd_movimentos', 'mean'),
        movimentos_desvio=('qtd_movimentos', 'std')
    ).reset_index()

    print("\n--- DF_RESUMO_ALPHAS ---")
    print(df_resumo_alphas.to_string(index=False))

    # pesos_finais, resultados = treinar_agente(mapa_labirinto, alpha, gama, epsilon, acoes_disponiveis, limite_movimentos, posicao_inicial, posicao_saida, seed, qtd_episodios)

    # df_resultados = pd.DataFrame(resultados)

    # df_resultados["pontuacao_media"] = (df_resultados["pontuacao"].rolling(window=50, min_periods=1).mean())

    # df_resultados["movimentos_medios"] = (df_resultados["movimentos"].rolling(window=50, min_periods=1).mean())

    # df_resultados["taxa_sucesso"] = (df_resultados["chegou_saida"].astype(int).rolling(window=50, min_periods=1).mean()* 100)

    # plt.figure(figsize=(10, 5))
    # plt.plot(df_resultados["episodio"], df_resultados["pontuacao_media"], color="blue", linewidth=2, )
    # plt.title("Evolução da Pontuação Média (Média Móvel de 50 Episódios)")
    # plt.xlabel("Episódio")
    # plt.ylabel("Pontuação Média")
    # plt.grid(True, linestyle="--", alpha=0.6)
    # plt.show()

    # plt.figure(figsize=(10, 5))
    # plt.plot(df_resultados["episodio"], df_resultados["movimentos_medios"], color="orange", linewidth=2, )
    # plt.title("Evolução da Quantidade Média de Movimentos (Média Móvel de 50 Episódios)")
    # plt.xlabel("Episódio")
    # plt.ylabel("Movimentos Médios")
    # plt.grid(True, linestyle="--", alpha=0.6)
    # plt.show()

    # plt.figure(figsize=(10, 5))
    # plt.plot(df_resultados["episodio"], df_resultados["taxa_sucesso"], color="green", linewidth=2, )
    # plt.title("Evolução da Taxa de Sucesso (Média Móvel de 50 Episódios)")
    # plt.xlabel("Episódio")
    # plt.ylabel("Sucesso (%)")
    # plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0f}%"))
    # plt.grid(True, linestyle="--", alpha=0.6)
    # plt.show()
    
    # print("\n--- FIM DO TREINAMENTO ---")
    # print(f"Quantidade de registros: {len(resultados)}")
    # print(f"Pesos finais: {pesos_finais}")

    # gerador_avaliacao = np.random.default_rng(123)

    # pontuacao, movimentos, sucesso, caminho = avaliar_agente(pesos_finais, mapa_labirinto, posicao_inicial, posicao_saida, acoes_disponiveis, gerador_avaliacao, limite_movimentos)

    # print("\n--- AVALIAÇÃO SEM EXPLORAÇÃO ---")
    # print("Pontuação:", pontuacao)
    # print("Movimentos:", movimentos)
    # print("Chegou à saída:", sucesso)
    # print("Primeiras 25 posições:", caminho[:25])

    


--- INÍCIO DA COMPARAÇÃO DOS ALPHAS ---

--- RESULTADOS DA COMPARAÇÃO ---
 alpha  seed  episodios  pontuacao_acumulada  qtd_movimentos  chegou_saida
 0.005    42       1000                 -150             150         False
 0.005    43       1000                 -150             150         False
 0.005    44       1000                 -150             150         False
 0.010    42       1000                   17              14          True
 0.010    43       1000                 -150             150         False
 0.010    44       1000                   17              14          True
 0.050    42       1000                   17              14          True
 0.050    43       1000                   12              10          True
 0.050    44       1000                   12              10          True

Quantidade de registros: 9

--- DF_RESUMO_ALPHAS ---
 alpha  taxa_sucesso  pontuacao_media  pontuacao_desvio  movimentos_medios  movimentos_desvio
 0.005      0.000000      -1